<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
《<a href="https://mng.bz/lZ5B">从零构建推理模型</a>》一书的配套代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库： <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第 2 章：使用预训练 LLM 生成文本

本笔记中使用的包：

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # 由 reasoning_from_scratch 使用
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F01_raschka.webp?1" width="700px">

&nbsp;
## 2.1 大型语言模型文本生成入门

- 本节不含代码
- LLM 如何生成文本？
- 本章是设置章节：配置贯穿全书的编码环境和 LLM
- 同时编写将在后续章节使用和扩展的文本生成函数

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F02_raschka.webp?1" width="500px">

- LLM（以及一般的神经网络）流程图通常自上而下绘制并阅读。

&nbsp;
## 2.2 设置编码环境

- 如果你在阅读这本书，很可能已经用 Python 开发过
- 如果你已经拥有 Python 3.10 或更高版本的环境，用 `pip` 安装依赖是最简单的方式：

In [ ]:
!git clone https://github.com/allen-shaw/reasoning-from-scratch-CN.git
%pip install -r /content/reasoning-from-scratch-CN/requirements.txt

- 在本章中，也可以手动安装这些依赖：

In [ ]:
#!pip install torch>=2.10.0 tokenizers>=0.22.2 reasoning-from-scratch

- 我更推荐使用广受推崇的 [uv](https://docs.astral.sh/uv/) Python 包与项目管理器
- 安装 `uv` 时,请依照官方站点的操作系统安装指南进行: https://docs.astral.sh/uv/getting-started/installation/
- 接着，克隆 GitHub 仓库：

In [ ]:
#!git clone --depth 1 https://github.com/rasbt/reasoning-from-scratch.git

- 如果没有安装 `git` 也可以从 Manning 网站手动下载源码，或者直接点击: https://github.com/rasbt/reasoning-from-scratch/archive/refs/heads/main.zip (下载后再解压)


- 在终端中进入 `reasoning-from-scratch` 文件夹
- 仓库包含 `.python-version` 文件，因此 `uv` 默认会使用与 PyTorch 兼容的 Python 版本
- 运行 `uv run jupyter lab` 启动 JupyterLab，然后打开空白 notebook 或本章的 notebook
- 此命令还会创建本地虚拟环境（通常位于 `.venv/`），并自动安装 `reasoning-from-scratch` 文件夹中 `pyproject.toml` 定义的所有依赖


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F03_raschka.webp?1" width="500px">

- 如需更多安装细节或其它选项，可参考 [../02_setup-tips/python-instructions.md](../02_setup-tips/python-instructions.md) 


&nbsp;
## 2.3 了解硬件需求与建议


- 如果你刚接触 PyTorch，推荐先阅读我的教程[PyTorch in One Hour: From Tensors to Training Neural Networks on Multiple GPUs](https://sebastianraschka.com/teaching/pytorch-1h/) 
- 如果你按照上一节操作过，PyTorch 已经安装好了。
- 手动检查一下当前 PyTorch 是否支持 GPU，并确认你的机器支持哪些:


In [ ]:
import torch


print(f"PyTorch version {torch.__version__}")

if torch.cuda.is_available():
    print(f"CUDA/ROCm GPU: {torch.cuda.get_device_name(0)}")

elif torch.xpu.is_available():
    print(f"Intel GPU: {torch.xpu.get_device_name(0)}")

elif torch.backends.mps.is_available():
    print("Apple Silicon GPU")

else:
    print("Only CPU")

- 不同章节会自动选择可用的 NVIDIA GPU；如果没有，就落到 CPU（或本章建议的 Apple Silicon GPU）。
- 第 2–4 章在 CPU 上也能在可接受时间内跑完。
- 第 5–7 章在 CPU 上会非常慢，建议使用带 NVIDIA GPU 的机器（后续章节会详细说明资源需求）。
- 我个人偏好 [Lightning AI Studio](https://lightning.ai/)，注册并认证后有免费算力；另外 [Google Colab](https://colab.research.google.com/) 也是不错的选择。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F04_raschka.webp" width="500px">

- 如需云端算力推荐，可参阅  [../02_setup-tips/gpu-instructions.md](../02_setup-tips/gpu-instructions.md) 
- 目前暂时还不用 GPU，前几章在无 GPU 的硬件上就能顺利运行。

&nbsp;
## 2.4 为 LLM 准备输入文本


- 本节我们学习如何使用 tokenizer，将输入文本编码为 token ID 形式，以便送入 LLM。
- 同时也用 tokenizer 把 LLM 的输出解码回人类可读的文本。


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F05_raschka.webp?1" width="500px">

- 如前所述，本书不会从零实现 LLM 和 tokenizer，而是专注在已有 LLM/tokenizer 之上，从零构建推理方法。
- 我们会在下一节加载一个预训练 LLM，这一节先加载与之配套的 tokenizer。
- 我准备了 `reasoning_from_scratch` Python 包，提供基础 LLM 及其 tokenizer，底层借助[`tokenizers`](https://github.com/huggingface/tokenizers) 库实现。
- 这个包 `reasoning_from_scratch` 包含在本书的配套代码中，按照第 2.2 节的说明已经安装好了。


- 接下来我们下载 tokenizer 文件（这是 Qwen3 基础版 LLM 所用的 tokenizer，更多细节在下一节介绍）：


In [ ]:
from reasoning_from_scratch.qwen3 import download_qwen3_small

download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")

- 现在可以把 tokenizer 文件中的配置加载到 `Qwen3Tokenizer` 中:


In [ ]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer

tokenizer_path = Path("qwen3") / "tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

- 由于我们尚未加载 LLM，本节先做一次更简单的往返：把文本编码成 token ID，再把这些 ID 解码回字符串表示：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F06_raschka.webp" width="500px">

In [ ]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)
print(input_token_ids_list)

In [ ]:
for i in input_token_ids_list:
    print(f"{i} --> {tokenizer.decode([i])}")

In [ ]:
text = tokenizer.decode(input_token_ids_list)
print(text)

- `Qwen3Tokenizer` 的词表大约包含 15.1 万个不同的词元。


- 更多关于分词的资料：
  - [Build a Large Language Model (from Scratch)](https://mng.bz/M96o) 第 2 章
  - [从零实现字节对编码（BPE）分词器](https://sebastianraschka.com/blog/2025/bpe-from-scratch.html)


&nbsp;
## 2.5 加载预训练模型

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F07_raschka.webp" width="500px">

- 如上一节加载 tokenizer 时提到的，本书选用 Qwen3 0.6B 作为基础模型；在对比多种开源权重后，选择 Qwen3 的原因是：
  - 截至写作时，Qwen3 在建模性能上是开源权重中的领先者；
  - Qwen3 0.6B 的显存占用比 Llama 3 1B 更少；
  - 提供了基础版（我们以它为底开发推理方法）和官方推理版，后者可作为对照参考。

- 注意 “Qwen3” 的正式写法没有空格，而 “Llama 3” 有空格。
- 本书秉承 “from-scratch” 的精神，使用我基于纯 PyTorch 重写的 Qwen3，不依赖任何外部 LLM 库，并且与原版 Qwen3 权重兼容。
- 不过本书不会展开讲解这份 Qwen3 实现(那又会是另一本书，类似我的 [Build A Large Language Model (From Scratch)](https://github.com/rasbt/LLMs-from-scratch),而是专注于在这个基础模型上实现推理方法。
- Qwen3 模型代码见附录 C。
- 附录 D 介绍如何加载推理版和更大的 Qwen3 模型。
- 更多细节可参考 Qwen3 的 [GitHub repository](https://github.com/QwenLM/Qwen3) 和 [technical report](https://arxiv.org/abs/2505.09388) 


- 该模型特意设计得较小（但能力依然很强），以便在消费级硬件上运行
- 它可以在 CPU、NVIDIA GPU（`"cuda"`）、Apple Silicon GPU（`"mps"`）和 Intel GPU（`"xpu"`）上正常运行；本章后面会进一步讨论相应的性能取舍


In [ ]:
def get_device(enable_tensor_cores=True):
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using NVIDIA CUDA GPU")
        
        if enable_tensor_cores:
            major, minor = map(int, torch.__version__.split(".")[:2])
            if (major, minor) >= (2, 9):
                torch.backends.cuda.matmul.fp32_precision = "tf32"
                torch.backends.cudnn.conv.fp32_precision = "tf32"
            else:
                torch.backends.cuda.matmul.allow_tf32 = True
                torch.backends.cudnn.allow_tf32 = True

    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")

    elif torch.xpu.is_available():
        device = torch.device("xpu")
        print("Using Intel GPU")

    else:
        device = torch.device("cpu")
        print("Using CPU")

    return device

device = get_device()

- 建议首次运行时使用 `"cpu"`，因此下面将设备硬编码为 CPU：


In [ ]:
# 建议：首次运行时使用 CPU
device = torch.device("cuda")

- 接着下载预训练权重文件，大小大约 1.5 GB：

In [ ]:
download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

- 下图展示了我们加载的 Qwen3 0.6B 模型的架构供熟悉 LLM 结构的读者参考，但请注意，本书并不要求理解这些细节，因为我们不会修改模型本身，而是后续在其上叠加推理方法。


- 我在本仓库内的 [reasoning-from-scratch](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/qwen3.py) Python 包里，从零实现了 Qwen3 的模型架构；源码也收录在附录 C。不过这只是给好奇的读者作为附加材料，理解这些内部细节并非阅读本书其它内容的必要条件。


In [ ]:
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B

model_path = Path("qwen3") / "qwen3-0.6B-base.pth"

model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))

model.to(device)

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F08_raschka.webp" width="500px">

&nbsp;
## 2.6 理解 LLM 顺序生成文本的过程

- 本节会编写一个简单的封装函数，方便调用 LLM 来生成文本（第 4 章会在此基础上扩展更多功能）。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F09_raschka.webp?1" width="600px">

- LLM 是一次生成一个词：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F10_raschka.webp?2" width="700px">

- 上图只是简化示意，只显示了新生成的词；下面的图放大了第一轮迭代的细节。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F11_raschka.webp" width="3b00px">

In [ ]:
example = torch.tensor([1, 2, 3]) 
print(example)
print(example.unsqueeze(0))

In [ ]:
example = torch.tensor([[1, 2, 3]]) 
print(example)
print(example.squeeze(0))

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F11_raschka.webp?2" width="500px">

In [ ]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)
print(f"Number of input tokens: {len(input_token_ids_list)}, input token ids {input_token_ids_list}")

input_tensor = torch.tensor(input_token_ids_list)
input_tensor_fmt = input_tensor.unsqueeze(0).to(device)
print(f"input_tensor: {input_tensor}, input_tensor_fmt: {input_tensor_fmt}, input.shape: {input_tensor.shape}")


with torch.inference_mode():
    output_tensor = model(input_tensor_fmt)

output_tensor_fmt = output_tensor.squeeze(0)
print(f"Formatted Output tensor shape: {output_tensor_fmt.shape}")

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F12_raschka.webp" width="700px">

In [ ]:
last_token = output_tensor_fmt[-1]
print(last_token)

In [ ]:
print(torch.argmax(last_token, dim=-1, keepdim=True))

In [ ]:
print(tokenizer.decode([20286]))

In [ ]:
example = torch.tensor([-2, 1, 3, 1])
print(torch.max(example))
print(torch.argmax(example))

&nbsp;
## 2.7 编写一个最简文本生成函数

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F13_raschka.webp" width="700px">

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F14_raschka.webp?2" width="700px">

- `generate_text_basic_stream` 函数实现了这一顺序文本生成过程：


In [ ]:
@torch.inference_mode()
def generate_text_basic_stream(
    model,
    token_ids,
    max_new_tokens, 
    eos_token_id=None
):
    model.eval()

    for _ in range(max_new_tokens):
        out = model(token_ids)[:, -1]
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        # 遇到序列结束词元时停止
        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        yield next_token  # 每生成一个词元就立即产出
        
        token_ids = torch.cat([token_ids, next_token], dim=1)

- 下面用它针对简单提示词 `"Explain large language models in a single sentence."` 生成一段最多 100 个词元的回复，观察其工作方式（后续章节才会进入推理部分）
- 以下代码运行较慢，视计算机性能而定，可能需要 1–3 分钟；后续小节会对它进行加速


In [ ]:
prompt = "Explain large language models in a single sentence."
input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device=device
    ).unsqueeze(0)
max_new_tokens = 100


for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True  # 禁用缓冲，使词元实时输出
    )

- 可以看到 LLM 基本遵循了指令，但在 `<|endoftext|>`之后回复开始跑题或胡言乱语；这个 token 是训练时用来分隔不同文档的分隔符。
- 实际使用 LLM 时，我们希望它在遇到该 token 后立刻停止生成。

In [ ]:
print(tokenizer.encode("<|endoftext|>"))

- 为方便使用，end-of-sequence（eos）这个 token 的 ID 被存成 tokenizer 的属性:

In [ ]:
print(tokenizer.eos_token_id)

- 可以用它告诉 LLM（更准确地说，是告诉 `generate_text_basic_stream` 函数）何时停止生成文本。


In [ ]:
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id  # 使用 EOS 词元
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

- 上面那段回答是在 CPU 上运行时得到的，具体输出可能会因为设备不同而略有差异。

- 在结束本节并介绍如何提速之前，先实现一个简单的基准函数，用来记录计算性能。

In [ ]:
import warnings

def generate_stats(output_token_ids, tokenizer, start_time,
                   end_time):
    total_time = end_time - start_time
    print(f"\n\nTime: {total_time:.2f} sec")
    print(f"{int(output_token_ids.numel() / total_time)} tokens/sec")

    for name, backend in (("CUDA", getattr(torch, "cuda", None)),
                          ("XPU", getattr(torch, "xpu", None))):
        if backend is not None and backend.is_available():

            # 检查实际是否使用了此后端
            device_type = output_token_ids.device.type
            if device_type != name.lower():
                warnings.warn(
                    f"{name} is available but tensors are on "
                    f"{device_type}. Memory stats may be 0."
                )
    
            # 如果支持则进行同步（对异步后端很重要）
            if hasattr(backend, "synchronize"):
                backend.synchronize()
            
            max_mem_bytes = backend.max_memory_allocated()
            max_mem_gb = max_mem_bytes / (1024 ** 3)
            print(f"Max {name} memory allocated: {max_mem_gb:.2f} GB")
            backend.reset_peak_memory_stats()

In [ ]:
import time

start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # 收集生成的词元

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

&nbsp;
## 2.8 通过 KV 缓存加速推理

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F15_raschka.webp?2" width="700px">

- 本书的代码更重视可读性，真正要优化可以单独写一本书。
- 这里我们介绍一个工程技巧 “KV 缓存”（KV 指 LLM 注意力机制里的 key 和 value）。
- 不熟悉这些术语也没关系，只要知道可以缓存每次迭代会重复使用的中间值，就能显著提速。


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F16_raschka.webp" width="700px">

- 若想进一步了解 KV 缓存的工作原理，请参阅我的文章 [Understanding and Coding the KV Cache in LLMs from Scratch](https://magazine.sebastianraschka.com/p/coding-the-kv-cache-in-llms)
- 下面是使用 KV 缓存的 `generate_text_basic_stream` 修改版


In [ ]:
from reasoning_from_scratch.qwen3 import KVCache

@torch.inference_mode()
def generate_text_basic_stream_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None
):
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])  # 新增
    model.reset_kv_cache()                           # 新增

    out = model(token_ids, cache=cache)[:, -1]
    for _ in range(max_new_tokens):
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        yield next_token
        out = model(next_token, cache=cache)[:, -1]

- 用法跟之前几乎一样：

In [ ]:
start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # 收集生成的词元

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

- 可以看到速度快了好几个数量级（在 Mac Mini M4 CPU 上从 4 tokens/sec 提升到 28 tokens/sec）。

&nbsp;
## 2.9 通过 PyTorch 模型编译加速推理

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F17_raschka.webp?2" width="600px">

- 另一种可以显著加速模型推理（文本生成）的技术是使用 `torch.compile`
- 用法很简单，只需对模型调用 `torch.compile`（其他选项参见[文档](https://docs.pytorch.org/docs/stable/torch.compiler_api.html)）


In [ ]:
major, minor = map(int, torch.__version__.split(".")[:2])
if (major, minor) >= (2, 8):
    # 这可以避免再次触发模型重新编译 
    # 适用于 PyTorch 2.8 及更高版本
    # 当模型包含 self.pos = self.pos + 1 之类代码时
    torch._dynamo.config.allow_unspec_int_on_nn_module = True

model_compiled = torch.compile(model)

# 如果在 "mps" 设备上使用 torch.compile 时遇到 InductorError，
# 请确保使用 PyTorch 2.9 或更高版本

---

**Windows 注意事项 1**

- 在 Windows 上进行编译可能比较棘手
- `torch.compile()` 使用 Inductor 即时编译内核，因此需要可用的 C/C++ 工具链
- 对于 CUDA，Inductor 还依赖 Triton；可以通过社区软件包 `triton-windows` 获取
  - 如果看到 `cl not found`，请[安装带“C++ 工作负载”的 Visual Studio Build Tools](https://learn.microsoft.com/en-us/cpp/build/vscpp-step-0-installation?view=msvc-170)，并在“x64 Native Tools”命令提示符中运行 Python
  - 如果使用 CUDA 时看到 `triton not found`，请安装 `triton-windows`（例如 `uv pip install "triton-windows<3.4"`）
- 对于 CPU，有读者进一步建议遵循这份 [Windows 版 PyTorch Inductor 指南](https://docs.pytorch.org/tutorials/unstable/inductor_windows.html)
  - 安装 Visual Studio 2022 时务必安装英文语言包，以避免 UTF-8 错误
  - 还请注意，代码需要通过“Visual Studio 2022 Developer Command Prompt”运行，而不是在 notebook 中运行
- 如果设置过程过于棘手，可以跳过编译；**编译是可选项，所有代码示例在不编译时也能正常工作**

**Windows 注意事项 2**

- 有读者报告，在 Windows 上使用默认设置运行 `torch.compile` 没有加速；但使用 `"max-autotune"` 模式时获得了 2 倍加速：`torch.compile(model, mode="max-autotune")`

---


- 由于第一次运行要做编译和优化，可能比较慢，所以会把生成过程重复多次。
- 先从不使用缓存的版本开始（可能会有些慢，耗时几分钟）。

In [ ]:
for i in range(3):

    start_time = time.time()
    generated_ids = []
    
    for token in generate_text_basic_stream(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True
        )
    
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)  # 收集生成的词元
    
    end_time = time.time()
    

    if i == 0:
        print("\n\nWarm-up run")
    else:
        print(f"\n\nTimed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

    print(f"\n{30*'-'}\n")

- 如上所示，速度大约 5 tokens/sec，比之前 4 tokens/sec 只快一点点。
- 接下来看看启用 KV 缓存的版本表现如何。

In [ ]:
for i in range(3):
    
    start_time = time.time()
    generated_ids = []
    
    for token in generate_text_basic_stream_cache(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True
        )
    
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)  # 收集生成的词元
    
    end_time = time.time()

    if i == 0:
        print("\n\nWarm-up run")
    else:
        print(f"\n\nTimed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(
        output_token_ids_tensor, tokenizer, start_time, end_time
    )

    print(f"\n{30*'-'}\n")

- 可以看到编译后带来了显著的 2 倍提速（从 30 tokens/sec 提升到 64 tokens/sec）。
- 下表列出了更多测量结果。

| 模型      | 模式              | 硬件             | 词元/秒    | GPU 显存（VRAM） |
|------------|-------------------|----------------------|---------------|-------------------|
| Qwen3模型 | 常规模式           | Mac Mini M4 CPU      | 5             | -                 |
| Qwen3模型 | 常规模式（已编译）  | Mac Mini M4 CPU      | 5             | -                 |
| Qwen3模型 | KV 缓存          | Mac Mini M4 CPU      | 29            | -                 |
| Qwen3模型 | KV 缓存（已编译） | Mac Mini M4 CPU      | 68            | -                 |
|            |                   |                      |               |                   |
| Qwen3模型 | 常规模式           | Mac Mini M4 GPU      | 27            | -                 |
| Qwen3模型 | 常规模式（已编译）  | Mac Mini M4 GPU      | 43            | -                 |
| Qwen3模型 | KV 缓存          | Mac Mini M4 GPU      | 41            | -                 |
| Qwen3模型 | KV 缓存（已编译） | Mac Mini M4 GPU      | 71            | -                 |
|            |                   |                      |               |                   |
| Qwen3模型 | 常规模式           | NVIDIA H100 GPU      | 51            | 1.55 GB           |
| Qwen3模型 | 常规模式（已编译）  | NVIDIA H100 GPU      | 164           | 1.81 GB           |
| Qwen3模型 | KV 缓存          | NVIDIA H100 GPU      | 48            | 1.52 GB           |
| Qwen3模型 | KV 缓存（已编译） | NVIDIA H100 GPU      | 141           | 1.81 GB           |
|            |                   |                      |               |                   |
| Qwen3模型 | 常规模式           | NVIDIA DGX Spark GPU | 74            | 1.53 GB           |
| Qwen3模型 | 常规模式（已编译）  | NVIDIA DGX Spark GPU | 103           | 1.49 GB           |
| Qwen3模型 | KV 缓存          | NVIDIA DGX Spark GPU | 68            | 1.47 GB           |
| Qwen3模型 | KV 缓存（已编译） | NVIDIA DGX Spark GPU | 98            | 1.47 GB           |

- 上述 NVIDIA DGX Spark 使用的是 GB10（Blackwell）GPU。
- 需要注意，我们所有示例都只用了单条提示（即 batch size = 1）；如果想了解批量推理，请参见附录 E。

&nbsp;
## 总结

- 本节没有代码。